# RAGAS Metrics Deep Dive

**Understanding How RAG Evaluation Metrics Work Internally**

**LangChain 1.0.5+ | RAGAS 0.3.9+ | Mixed Level Class**

---

## Objectives

1. **Understand the internal calculation process** for each of the 6 core RAGAS metrics
2. **See intermediate outputs** like extracted claims, generated questions, and identified entities
3. **Learn to interpret scores with confidence** using threshold guidelines
4. **Debug evaluation issues** by understanding what each metric actually measures

---

## 📚 What Makes This Notebook Different?

While Notebook 10 covers **how to use** RAGAS metrics, this notebook goes deeper into **how they work internally**:

| Notebook 10 | This Notebook (12) |
|-------------|--------------------|
| Run evaluation and get scores | See how scores are calculated step-by-step |
| Use metrics as black boxes | Understand intermediate outputs |
| Focus on results | Focus on the calculation process |

---

## 🔢 The 6 Metrics We'll Explore

| # | Metric | Evaluates | Key Question |
|---|--------|-----------|-------------|
| 1 | **Faithfulness** | Generator | Is the answer grounded in context? |
| 2 | **Answer Relevancy** | Generator | Does the answer address the question? |
| 3 | **Context Precision** | Retriever | Are relevant chunks ranked at the top? |
| 4 | **Context Recall** | Retriever | Did we retrieve all necessary information? |
| 5 | **Context Entity Recall** | Retriever | Did we retrieve all important entities? |
| 6 | **Noise Sensitivity** | System | Does noise cause wrong answers? |

---

## 🔰 Section 1: Setup & Environment

Let's set up our environment with all required imports and configure our LLM/Embedding models.

In [1]:
# Environment Setup
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# Verify API key
if os.getenv("OPENAI_API_KEY"):
    print("✅ OPENAI_API_KEY found")
else:
    print("❌ OPENAI_API_KEY not found - please set it in your .env file")

✅ OPENAI_API_KEY found


In [2]:
# Core Imports

# Standard library
import numpy as np
import pandas as pd
import asyncio
import json

# LangChain components
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# RAGAS components
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ContextEntityRecall,
    NoiseSensitivity
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("✅ All imports successful")

✅ All imports successful


In [3]:
# Initialize LLM and Embeddings

# Initialize base models  (OPENAI)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Wrap for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ LLM initialized: gpt-4o-mini")
print("✅ Embeddings initialized: text-embedding-3-small")
print("✅ RAGAS wrappers ready")

✅ LLM initialized: gpt-4o-mini
✅ Embeddings initialized: text-embedding-3-small
✅ RAGAS wrappers ready


### Initialize base models (GEMINI)
```bash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embeddings= GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
```


In [4]:
# Helper function for running async code in Jupyter

def run_async(coro):
    """Helper to run async code in Jupyter notebooks"""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # We're in Jupyter with an existing loop
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(coro)
        else:
            return asyncio.run(coro)
    except RuntimeError:
        return asyncio.run(coro)

print("✅ Async helper ready")

✅ Async helper ready


---

# 🔰 Section 2: Faithfulness Deep Dive

## What Faithfulness Measures

**Faithfulness** checks if the generated answer *sticks to the facts* from the retrieved context. It detects **hallucinations** - when the LLM makes things up that aren't in the source material.

### 📖 Analogy

> Imagine you're a journalist writing a news story. Faithfulness checks whether everything you wrote can be traced back to your interview notes. If you add details that weren't in your notes, that's a problem!

### 🔧 How It Works (3 Steps)

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  Step 1:        │    │  Step 2:        │    │  Step 3:        │
│  Extract Claims │ -> │  Verify Each    │ -> │  Calculate      │
│  from Response  │    │  Against Context│    │  Score          │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

### 📐 Formula

$$\text{Faithfulness} = \frac{\text{Number of claims supported by context}}{\text{Total number of claims}}$$

## 2.1 Step 1: Manual Claim Extraction

Let's first see how RAGAS extracts claims from a response. We'll mimic this process manually.

In [5]:
# Define our test case

# The response we want to evaluate
test_response = "The first Super Bowl was held on January 15, 1967 in Los Angeles. It was a sunny day with clear skies."

# The context that was retrieved (source of truth)
test_context = [
    "The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles, California."
]

print("📝 Response to evaluate:")
print(f"   '{test_response}'")
print("\n📚 Retrieved context:")
print(f"   '{test_context[0]}'")

📝 Response to evaluate:
   'The first Super Bowl was held on January 15, 1967 in Los Angeles. It was a sunny day with clear skies.'

📚 Retrieved context:
   'The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles, California.'


In [6]:
# Manual claim extraction using LLM (mimicking RAGAS)

claim_extraction_prompt = ChatPromptTemplate.from_template("""
Given the following response, extract ALL factual claims as a numbered list.
Each claim should be a single, verifiable statement.

Response: {response}

Extract each factual claim:
""")

claim_chain = claim_extraction_prompt | llm | StrOutputParser()

extracted_claims_raw = claim_chain.invoke({"response": test_response})

print("🔍 STEP 1: Extracted Claims from Response")
print("=" * 50)
print(extracted_claims_raw)

🔍 STEP 1: Extracted Claims from Response
1. The first Super Bowl was held on January 15, 1967.
2. The first Super Bowl was held in Los Angeles.
3. It was a sunny day on January 15, 1967.
4. There were clear skies on January 15, 1967.


In [7]:
# Parse the claims into a list for verification

# For our analysis, let's define the claims explicitly
claims = [
    "The first Super Bowl was held on January 15, 1967",
    "The first Super Bowl was held in Los Angeles",
    "It was a sunny day",
    "There were clear skies"
]

print("📋 Claims to verify:")
for i, claim in enumerate(claims, 1):
    print(f"   {i}. {claim}")

📋 Claims to verify:
   1. The first Super Bowl was held on January 15, 1967
   2. The first Super Bowl was held in Los Angeles
   3. It was a sunny day
   4. There were clear skies


## 2.2 Step 2: Claim Verification

Now we verify each claim against the retrieved context. This is where hallucinations are detected!

In [8]:
# Manual claim verification (mimicking RAGAS)

verification_prompt = ChatPromptTemplate.from_template("""
Given the following context and claim, determine if the claim is SUPPORTED by the context.

Context: {context}

Claim: {claim}

Answer with:
- "SUPPORTED" if the claim can be verified from the context
- "NOT SUPPORTED" if the claim cannot be verified or contradicts the context

Also provide a brief explanation.

Verdict:
""")

verify_chain = verification_prompt | llm | StrOutputParser()

print("🔍 STEP 2: Verifying Each Claim Against Context")
print("=" * 60)

verification_results = []
for claim in claims:
    result = verify_chain.invoke({
        "context": test_context[0],
        "claim": claim
    })
    is_supported = "SUPPORTED" in result.upper() and "NOT SUPPORTED" not in result.upper()
    verification_results.append({
        "claim": claim,
        "supported": is_supported,
        "explanation": result
    })
    status = "✅" if is_supported else "❌"
    print(f"\n{status} Claim: '{claim}'")
    print(f"   Result: {result[:100]}..." if len(result) > 100 else f"   Result: {result}")

🔍 STEP 2: Verifying Each Claim Against Context

✅ Claim: 'The first Super Bowl was held on January 15, 1967'
   Result: SUPPORTED

Explanation: The context states that the First AFL-NFL World Championship Game was played...

✅ Claim: 'The first Super Bowl was held in Los Angeles'
   Result: SUPPORTED

Explanation: The context states that the First AFL-NFL World Championship Game, which is ...

❌ Claim: 'It was a sunny day'
   Result: NOT SUPPORTED

The context provides information about the date and location of the First AFL-NFL Wor...

❌ Claim: 'There were clear skies'
   Result: NOT SUPPORTED

The context provides information about the date and location of the First AFL-NFL Wor...


In [9]:
# Display verification results as a table

print("\n📊 Claim Verification Summary")
print("=" * 80)

df_verification = pd.DataFrame([
    {
        "Claim": r["claim"],
        "Supported?": "✅ Yes" if r["supported"] else "❌ No",
        "Reason": "Found in context" if r["supported"] else "HALLUCINATION - Not in context!"
    }
    for r in verification_results
])

print(df_verification.to_string(index=False))


📊 Claim Verification Summary
                                            Claim Supported?                          Reason
The first Super Bowl was held on January 15, 1967      ✅ Yes                Found in context
     The first Super Bowl was held in Los Angeles      ✅ Yes                Found in context
                               It was a sunny day       ❌ No HALLUCINATION - Not in context!
                           There were clear skies       ❌ No HALLUCINATION - Not in context!


## 2.3 Step 3: Calculate Faithfulness Score

In [10]:
# Manual faithfulness calculation

supported_count = sum(1 for r in verification_results if r["supported"])
total_claims = len(verification_results)

manual_faithfulness = supported_count / total_claims

print("🔢 STEP 3: Calculate Faithfulness Score")
print("=" * 50)
print(f"\n   Supported claims: {supported_count}")
print(f"   Total claims: {total_claims}")
print(f"\n   Formula: Faithfulness = {supported_count} / {total_claims}")
print(f"\n   📊 Manual Faithfulness Score: {manual_faithfulness:.2f}")

🔢 STEP 3: Calculate Faithfulness Score

   Supported claims: 2
   Total claims: 4

   Formula: Faithfulness = 2 / 4

   📊 Manual Faithfulness Score: 0.50


## 2.4 Verify with Actual RAGAS Metric

Now let's compare our manual calculation with the actual RAGAS Faithfulness metric!

In [11]:
# Run actual RAGAS Faithfulness metric

# Create sample in RAGAS format
faithfulness_sample = SingleTurnSample(
    user_input="When was the first Super Bowl?",
    response=test_response,
    retrieved_contexts=test_context
)

# Initialize and run the metric
faithfulness_metric = Faithfulness(llm=ragas_llm)

ragas_faithfulness = run_async(faithfulness_metric.single_turn_ascore(faithfulness_sample))

print("🔬 RAGAS Faithfulness Result")
print("=" * 50)
print(f"\n   Manual calculation:  {manual_faithfulness:.2f}")
print(f"   RAGAS metric score:  {ragas_faithfulness:.2f}")
print(f"\n   Difference: {abs(manual_faithfulness - ragas_faithfulness):.2f}")

🔬 RAGAS Faithfulness Result

   Manual calculation:  0.50
   RAGAS metric score:  0.50

   Difference: 0.00


## 2.5 Faithfulness Examples: Good vs Bad

Let's see how different types of responses score on Faithfulness.

In [12]:
# Compare different faithfulness scenarios

faithfulness_examples = [
    {
        "name": "Perfect Faithfulness (No hallucinations)",
        "response": "The first Super Bowl was played on January 15, 1967 at the Los Angeles Memorial Coliseum.",
        "context": ["The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum."]
    },
    {
        "name": "Partial Faithfulness (Some hallucinations)",
        "response": "The first Super Bowl was on January 15, 1967. The Green Bay Packers won 35-10 with Bart Starr as MVP.",
        "context": ["The First AFL-NFL World Championship Game was played on January 15, 1967."]
    },
    {
        "name": "Zero Faithfulness (Complete hallucination)",
        "response": "The first Super Bowl was held in Miami in 1970 and attracted over 100,000 spectators.",
        "context": ["The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum."]
    }
]

print("📊 Faithfulness Comparison: Different Scenarios")
print("=" * 70)

for example in faithfulness_examples:
    sample = SingleTurnSample(
        user_input="Tell me about the first Super Bowl",
        response=example["response"],
        retrieved_contexts=example["context"]
    )
    score = run_async(faithfulness_metric.single_turn_ascore(sample))
    
    print(f"\n🏷️  {example['name']}")
    print(f"   Response: '{example['response'][:80]}...'" if len(example['response']) > 80 else f"   Response: '{example['response']}'")
    print(f"   Score: {score:.2f}")

📊 Faithfulness Comparison: Different Scenarios

🏷️  Perfect Faithfulness (No hallucinations)
   Response: 'The first Super Bowl was played on January 15, 1967 at the Los Angeles Memorial ...'
   Score: 1.00

🏷️  Partial Faithfulness (Some hallucinations)
   Response: 'The first Super Bowl was on January 15, 1967. The Green Bay Packers won 35-10 wi...'
   Score: 0.33

🏷️  Zero Faithfulness (Complete hallucination)
   Response: 'The first Super Bowl was held in Miami in 1970 and attracted over 100,000 specta...'
   Score: 0.00


## 2.6 Score Interpretation Guide

| Score Range | Interpretation | Action |
|-------------|----------------|--------|
| **0.9 - 1.0** | Excellent - No or minimal hallucinations | ✅ Good to go |
| **0.7 - 0.9** | Good - Minor unsupported claims | ⚠️ Review edge cases |
| **0.5 - 0.7** | Concerning - Significant hallucinations | 🔧 Improve prompt/temperature |
| **< 0.5** | Poor - Most claims are hallucinated | 🚨 Major fixes needed |

---

# 🔰 Section 3: Answer Relevancy Deep Dive

## What Answer Relevancy Measures

**Answer Relevancy** checks if the answer *actually answers* the question asked. It doesn't care if the answer is factually correct - just whether it's relevant to the question.

### 📖 Analogy

> If someone asks "What's the capital of France?" and you answer "The Eiffel Tower is beautiful," your answer might be factually true but completely irrelevant to the question!

### 🔧 The "Reverse Engineering" Approach

RAGAS uses a clever technique: instead of directly comparing the answer to the question, it:

1. **Generates hypothetical questions** from the answer ("What questions would this be a good answer to?")
2. **Compares embeddings** of generated questions with the original question
3. **Calculates similarity** - if the generated questions are similar to the original, the answer is relevant!

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  Step 1:        │    │  Step 2:        │    │  Step 3:        │
│  Generate       │ -> │  Embed All      │ -> │  Calculate      │
│  Questions      │    │  Questions      │    │  Similarity     │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

### 📐 Formula

$$\text{Answer Relevancy} = \frac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(E_{generated_i}, E_{original})$$

## 3.1 Step 1: Hypothetical Question Generation

Let's see how RAGAS generates questions from an answer.

In [13]:
# Define our test case for relevancy

original_question = "When was the first Super Bowl?"
test_answer = "The first Super Bowl was held on January 15, 1967"

print("📝 Original Question:")
print(f"   '{original_question}'")
print("\n📝 Answer to Evaluate:")
print(f"   '{test_answer}'")

📝 Original Question:
   'When was the first Super Bowl?'

📝 Answer to Evaluate:
   'The first Super Bowl was held on January 15, 1967'


In [14]:
# Manual hypothetical question generation (mimicking RAGAS)

question_gen_prompt = ChatPromptTemplate.from_template("""
Given the following answer, generate exactly 3 different questions that this answer would be a good response to.
The questions should be varied but all answerable by this response.

Answer: {answer}

Generate 3 questions (one per line):
1.
2.
3.
""")

question_gen_chain = question_gen_prompt | llm | StrOutputParser()

generated_questions_raw = question_gen_chain.invoke({"answer": test_answer})

print("🔍 STEP 1: Generated Hypothetical Questions")
print("=" * 50)
print(generated_questions_raw)

🔍 STEP 1: Generated Hypothetical Questions
1. When was the inaugural Super Bowl played?  
2. What date marks the beginning of the Super Bowl history?  
3. Can you tell me when the first Super Bowl took place?  


In [15]:
# Parse generated questions (for our calculation)

# Manually define likely generated questions
generated_questions = [
    "When was the first Super Bowl held?",
    "What date was the inaugural Super Bowl?",
    "On what day did the first Super Bowl take place?"
]

print("📋 Questions for embedding comparison:")
print(f"   Original: '{original_question}'")
print("   Generated:")
for i, q in enumerate(generated_questions, 1):
    print(f"      {i}. '{q}'")

📋 Questions for embedding comparison:
   Original: 'When was the first Super Bowl?'
   Generated:
      1. 'When was the first Super Bowl held?'
      2. 'What date was the inaugural Super Bowl?'
      3. 'On what day did the first Super Bowl take place?'


## 3.2 Step 2: Embedding and Similarity Calculation

Now we compute embeddings and calculate cosine similarity.

In [16]:
# Define cosine similarity function

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print("✅ Cosine similarity function ready")
print("\n📐 Formula: cos(θ) = (A · B) / (||A|| × ||B||)")

✅ Cosine similarity function ready

📐 Formula: cos(θ) = (A · B) / (||A|| × ||B||)


In [17]:
# Calculate embeddings and similarities

print("🔍 STEP 2: Computing Embeddings and Similarities")
print("=" * 60)

# Get embedding for original question
original_embedding = embeddings.embed_query(original_question)
print(f"\n✅ Original question embedded (dim={len(original_embedding)})")

# Get embeddings for generated questions and calculate similarities
similarities = []
for i, gen_q in enumerate(generated_questions, 1):
    gen_embedding = embeddings.embed_query(gen_q)
    sim = cosine_similarity(original_embedding, gen_embedding)
    similarities.append(sim)
    print(f"\n   Question {i}: '{gen_q}'")
    print(f"   Similarity to original: {sim:.4f}")

🔍 STEP 2: Computing Embeddings and Similarities

✅ Original question embedded (dim=1536)

   Question 1: 'When was the first Super Bowl held?'
   Similarity to original: 0.9353

   Question 2: 'What date was the inaugural Super Bowl?'
   Similarity to original: 0.8009

   Question 3: 'On what day did the first Super Bowl take place?'
   Similarity to original: 0.9063


## 3.3 Step 3: Calculate Final Score

In [19]:
# Calculate answer relevancy score

manual_relevancy = np.mean(similarities)

print("🔢 STEP 3: Calculate Answer Relevancy Score")
print("=" * 50)
print(f"\n   Similarities: {[f'{s:.4f}' for s in similarities]}")
print(f"   Formula: Average of similarities")
print(f"\n   ({' + '.join([f'{s:.4f}' for s in similarities])}) / {len(similarities)}")
print(f"\n   📊 Manual Answer Relevancy: {manual_relevancy:.4f}")

🔢 STEP 3: Calculate Answer Relevancy Score

   Similarities: ['0.9353', '0.8009', '0.9063']
   Formula: Average of similarities

   (0.9353 + 0.8009 + 0.9063) / 3

   📊 Manual Answer Relevancy: 0.8808


## 3.4 Verify with Actual RAGAS Metric

In [20]:
# Run actual RAGAS Answer Relevancy metric

relevancy_sample = SingleTurnSample(
    user_input=original_question,
    response=test_answer,
    retrieved_contexts=["The First AFL-NFL World Championship Game was played on January 15, 1967."]
)

relevancy_metric = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

ragas_relevancy = run_async(relevancy_metric.single_turn_ascore(relevancy_sample))

print("🔬 RAGAS Answer Relevancy Result")
print("=" * 50)
print(f"\n   Manual calculation:  {manual_relevancy:.4f}")
print(f"   RAGAS metric score:  {ragas_relevancy:.4f}")

🔬 RAGAS Answer Relevancy Result

   Manual calculation:  0.8808
   RAGAS metric score:  0.9353
